In [44]:
%reset -f 
# resetting stored variables in case there's something weird cached
from build123d import *
from ocp_vscode import *
import cadquery as cq
import time
import math
from library.tools import *
import sys
from dataclasses import dataclass, field
import bd_warehouse.thread, bd_warehouse.fastener 

ALL UNITS IN MM
DON'T @ ME

In [45]:
%reload_ext ocp_vscode
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [46]:

print(f"Python executable: {sys.executable}")
print(f"OCP-vscode location: {sys.modules.get('ocp_vscode', 'Not found')}")

Python executable: c:\Users\Kaoti\parthenon\.venv\Scripts\python.exe
OCP-vscode location: <module 'ocp_vscode' from 'c:\\Users\\Kaoti\\parthenon\\.venv\\Lib\\site-packages\\ocp_vscode\\__init__.py'>


Toy parts: 

In [47]:
reset_show()
box = cq.Workplane().box(1,2,1).edges().chamfer(0.4)
show_object(box, name="Chamfered Box", options={"alpha": 0.8})

c


In [48]:
sphere = cq.Workplane().sphere(0.6)
reset_show()
show_object(sphere)

c


In [49]:
reset_show()
box = Box(1, 2, 1)
show(box, reset_camera=Camera.RESET)
print(f"Box created: {box}")  # Should show object details
# Does anything appear in the viewer panel?

c
Box created: Box at 0x1208cbffd90, label(), #children(0)


Pseudocode: 

Construct primitives (Initial GT frame circle,)
Orient primitives
Translate primitives
Fuse primitives



Actual code: 

In [50]:

reset_show()

.translate(x,y,z)
reset_camera=Camera.RESET
add(mypart)
reset_show
Cylinder(radius, height)

In [51]:
# Parameters for basis cylinder

# Parameters for cylindrical shapes for the basis cylidner
chamberRadius1=27.5/2 
chamberRadius2=30.150/2 - 0.025  - 0.05 #0.025mm thicken step in arbor npx backbone. built in here to the radius
chamberRadius3=36/2
chamberGripOuterHeight=10
chamberGripInnerHeight=5

# Cutaway triangle parameters
cutTrianglePrimaryAngle=123.547500 # This is the cutout angle in the basis cylinder
cutTriangleSecondaryAngle=88.546500 # This is the cutout angle relative to the topmost throughhole axis
cutTriangleHeight=4
triangleAngle = 123
triangleThickness = 4
vertices1 = (0,0,0)
vertices2 = (-2*chamberRadius3,0,0)
vertices3 = (-3*chamberRadius3*(math.cos(math.radians(triangleAngle))), -3*chamberRadius3*(math.sin(math.radians(triangleAngle))),0)

# Throughhole parameters
throughHoleAngle=90.0 # This is the angle between the throughholes
throughHoleRadius=2.500/2 # This is the radius of the throughholes

# Directional indicator triangle parameters
indicatorTriangleAngle = -1 * (33.456000 + throughHoleAngle)
indicatorTriangleSideLength = 3.46410
indicatorTriangleInternalAngles = 180/3
indicatorOffset = 3.0 / 2
indicatorHorizontalDisplacement = 14.50000
indicatorTotalYDisplacement = indicatorHorizontalDisplacement + indicatorOffset + (chamberRadius3 - chamberRadius1)/2
indicatorTriangleThickness = 0.50000


In [52]:
# Basis cylinder

# Cylinder parts

# This is a movement vector which moves the cylinders where I want them
cylinderTranslationVector=(0,0,(chamberGripOuterHeight-chamberGripInnerHeight)/2)

# Constructing initial primitive cylinders
cylinder1 = Cylinder(radius=chamberRadius1, height=chamberGripOuterHeight)
cylinder2 = Cylinder(radius=chamberRadius2, height=chamberGripInnerHeight).translate(cylinderTranslationVector)
cylinder3 = Cylinder(radius=chamberRadius3, height=chamberGripOuterHeight)

# Throughhole: Primitive cylinders and translation and rotation vectors to move them
throughHole1TranslationVector=(0,(chamberRadius2+chamberRadius3)/2,1.15100+throughHoleRadius,0)
throughHole1RotationVector=(1,0,0)

throughHole2TranslationVector=((chamberRadius2+chamberRadius3)/2,0,1.15100+throughHoleRadius)
throughHole2RotationVector=(0,1,0)

throughHole1Cylinder = Cylinder(radius=throughHoleRadius, height=chamberRadius1/2).rotate(
    axis = Axis(Vector(0,0,0), throughHole1RotationVector),
    angle=90
).translate(throughHole1TranslationVector)

throughHole2Cylinder = Cylinder(radius=throughHoleRadius,height=chamberRadius1/2).rotate(
    axis = Axis(Vector(0,0,0), throughHole2RotationVector),
    angle = 90
).translate(throughHole2TranslationVector)

# Cutaway Triangle primitives, movement
with BuildPart() as bp:
    with BuildSketch():
        Polygon(vertices1, vertices2, vertices3, align=None)
    extrude(amount=triangleThickness)

basisTriangle = bp.part.translate((0,0,(chamberGripOuterHeight/2 - triangleThickness)))

# Directional Indicator Triangle primitives
with BuildPart() as bp:
    with BuildSketch():
        Polygon((-indicatorOffset,0,0), (indicatorOffset,0,0), (0,indicatorOffset*2,0), align=None)
    extrude(amount=indicatorTriangleThickness)

indicatorTriangleTranslationVector = (0,indicatorHorizontalDisplacement,-(indicatorTriangleThickness + chamberGripOuterHeight)/2)
indicatorTriangleRotationVector = (0,0,1)

indicatorTriangle = bp.part.translate(indicatorTriangleTranslationVector).rotate(
    axis = Axis(Vector(0,0,0), indicatorTriangleRotationVector),
    angle = indicatorTriangleAngle
)



In [53]:
# We need 45 degree screw holes, so this is where we're going to build those for later addition to the basis cylinder
# Note: In Anna's original CAD the screw hole does not infiltrate the GT frame proper, but rather stays inside the 45 degree block.
# First we have to use the +Z axis and its corresponding indicator triangle as a reference to offset the location. Essentially we're just building a rotation vector. 

screwHoleZOffset = 15 # degrees
screwHoleRotationVector = [0,0,screwHoleZOffset]

For subtracting items as masks:     add(cylinder1, mode=Mode.SUBTRACT)

In [54]:
# Building the whole basis cylinder

reset_show()

# Doing the initial construction of the basis cylinder
with BuildPart() as basisCylinder:
    add(cylinder3) # adding outermost cylinder
    add(cylinder1, mode=Mode.SUBTRACT) # Carving at outermost cylinder
    add(cylinder2, mode=Mode.SUBTRACT) # Carving again
    add(throughHole1Cylinder, mode=Mode.SUBTRACT) # Cutting throughholes
    add(throughHole2Cylinder, mode=Mode.SUBTRACT)
    add(basisTriangle, mode=Mode.SUBTRACT) # Adding the cutaway to accomodate skull structure
    add(indicatorTriangle) # adding indicator triangle for orienteering purposes

# Flipping the cylinder, since I decided building it upside-down at an odd angle was a great idea
basisCylinderRotationVector = (1,0,0)
basisCylinder = basisCylinder.part.rotate(
    axis = Axis(Vector(0,0,0), basisCylinderRotationVector),
    angle = 180
)

# Rotating the cylinder so the indicator triangle points to +y
basisCylinderRotationVector = (0,0,1)
basisCylinder = basisCylinder.rotate(
    axis = Axis(Vector(0,0,0), basisCylinderRotationVector),
    angle = 180 - (33.456000 + throughHoleAngle) 
)

# Translating the cylinder down 5mm in y axis so it aligns correctly with the chamber mesh. See note in next code box.
basisCylinderTranslationVector = (0,0,-5)
basisCylinder = basisCylinder.translate(basisCylinderTranslationVector)

show(basisCylinder, reset_camera=Camera.RESET)

c


In [55]:
# Importing reference chamber mesh - Goliath Posterior V3 starting V3 
# this will be an if statement based on a chamber input field. That field will also need
# to adjust the basis cylinder we generate 
chamberMold = Mesher().read("chamberMoldMeshGoliathPosteriorV3.stl")[0]

# NOTE: MESHES CANNOT BE TRANSLATED. TRANSLATE OTHER OBJECTS AROUND THEM.

# Displaying the mesh and the cylinder both
reset_show()
show(chamberMold,basisCylinder,reset_camera=Camera.RESET)

cc


Immediately below this point: Probably need to integrate coordinates from SPOTS somehow. Likely, also need a least-squares fit line for the arbor, and some parameter permutation if we want a normal offset from the arbor line. 

In [56]:
# Retrieving / generating points for probe penetration

# Parameters
meshZ = -5
GTHoleDiameter = 0.675
recessDiameter = 3.1
GTExteriorDiameter = 2

# Penetration center points - these will be inputs which generate whole structure
penetration1 = (0,7.7, meshZ)
penetration2 = (0,2.475,meshZ)
penetration3 = (0,-1.775,meshZ)
penetration4 = (0,-7,meshZ)
points = (penetration1, penetration2, penetration3, penetration4)
circularLocations = []

# Alternative construction which leaves each point accessible
for x in points:
    with BuildSketch() as sk:
            with Locations(x):
                Circle(radius = GTExteriorDiameter, align=None)
    circularLocations.append(sk)
print(circularLocations)

# Everybody gets a diameter
with BuildSketch() as sk:
    with Locations(points):
        Circle(radius = 2)
        Circle(radius = GTHoleDiameter/2, mode = Mode.SUBTRACT)

# These sketches now exist
penetrationLocations = sk

# Displaying everything at once
show (chamberMold, basisCylinder, circularLocations, reset_camera = Camera.RESET)

[<build123d.build_sketch.BuildSketch object at 0x000001208CA0DB50>, <build123d.build_sketch.BuildSketch object at 0x000001208CA0CD10>, <build123d.build_sketch.BuildSketch object at 0x000001208CA0DF10>, <build123d.build_sketch.BuildSketch object at 0x000001208CA0DD90>]
----cccccc


In [57]:
# Mesh projected curves
# This projects circles down onto the mesh surface and adds interfaces onto list
projectedCurves = []
for x in circularLocations:
    wire = x.sketch.faces()[0].outer_wire()
    hits = wire.project_to_shape(chamberMold,direction=(0,0,-1))
    projectedCurves.append(hits)

print(projectedCurves)

show (chamberMold, basisCylinder, circularLocations, projectedCurves, reset_camera = Camera.RESET)

[[<build123d.topology.one_d.Wire object at 0x0000012097206E50>, <build123d.topology.one_d.Wire object at 0x000001208CA0A250>], [<build123d.topology.one_d.Wire object at 0x0000012097204DD0>, <build123d.topology.one_d.Wire object at 0x0000012097207250>], [<build123d.topology.one_d.Wire object at 0x0000012097206ED0>, <build123d.topology.one_d.Wire object at 0x0000012097206FD0>], [<build123d.topology.one_d.Wire object at 0x00000120972067D0>, <build123d.topology.one_d.Wire object at 0x00000120972065D0>]]
----cccccc


In [58]:
# Building surfaces from projected curves
# To clarify, this grabs the center point of each curve, and the slope at that point
# Then uses those slopes to construct planes. These planes are later used to contruct shafts

frameFaces = []

for x in circularLocations:
    face = x.sketch.faces()[0]
    center = face.center()
    axis = Axis((center.X, center.Y, chamberMold.bounding_box().max.Z), (0,0,-1))
    point, normal = chamberMold.find_intersection_points(axis)[-1]
    tiltedPlane = Plane(origin=point, z_dir = normal)
    frameFaces.append(face.located(Location(tiltedPlane)))

In [59]:
# Does it work? This visualizes the planes and the projected curves

show (frameFaces, basisCylinder, circularLocations, projectedCurves, reset_camera = Camera.RESET)

----ccccccccc


In [60]:
# Offsetting the projected curve faces to provide targets for extrusion limits
# This allows 2mm of space between the tissue surface and the the bottom of the shafts

offsetFrameFaces = []
zPlanarOffset = 2   

for x in frameFaces:
    plane1 = Plane(x)
    offsetPlane = Plane(origin = plane1.origin + (0,0,zPlanarOffset), x_dir=plane1.x_dir, z_dir = plane1.z_dir)
    offsetFrameFaces.append(offsetPlane)

show(offsetFrameFaces, basisCylinder, circularLocations, projectedCurves, reset_camera = Camera.RESET)


----ccccc


In [61]:
# Transforms the faces which the shafts start extruding on into planes so they can be useful
# Should probably look at this - not sure if this is redundant

startingOffsetPlanes = []
shaftHeight = 3.85 # This is a rough estimation based on Anna's onshape
shaftDiameter = 4
shaftRadius = shaftDiameter/2
innerShaftDiameter = 0.67500
innerShaftRadius = innerShaftDiameter/2

for x in offsetFrameFaces: # These are actually planes, not faces
    appendMe = Plane(origin = x.origin + (0,0,shaftHeight), x_dir = (1,0,0), z_dir = (0,0,1))
    startingOffsetPlanes.append(appendMe)

In [62]:
# Generating solids from bottom surface planes for later use.
# 

bottomSurfaceSolids = []

for x in offsetFrameFaces:
    with BuildPart() as cyl:
        with BuildSketch(x):
            Circle(radius = shaftDiameter)
        extrude(amount=1)
    bottomSurfaceSolids.append(cyl.part)

show(bottomSurfaceSolids, startingOffsetPlanes)

cccc


In [63]:
# Building circles on the projected curves
shaftList = []
throughHoles = []
shaftListWithThroughHoles = []
seams = []
j = 0

for i, x in enumerate(startingOffsetPlanes): 
    # First build the outer shaft
    with BuildPart() as shaft:
        with BuildSketch(x) as sk:
            Circle(radius=shaftRadius)
        extrude(until=Until.NEXT, target = bottomSurfaceSolids[i], dir=(0,0,-1))
    shaftList.append(shaft)

    # Next build the throughhole
    with BuildPart() as throughhole:
        with BuildSketch(x) as sk:
            Circle(radius = innerShaftRadius)
        extrude(until = Until.NEXT, target = bottomSurfaceSolids[i], dir=(0,0,-1))
    throughHoles.append(throughhole)

    # Now subtract the throughhole geometry from the shaft geometery
    with BuildPart() as shaftWithThroughHole:
        add(shaftList[i])
        add(throughHoles[i], mode = Mode.SUBTRACT)
    shaftListWithThroughHoles.append(shaftWithThroughHole)

# show the throughholes in red and the shafts in yellow for diagnostic purposes
show(shaftList, throughHoles, colors = ["Yellow", "Red"])


cccccccc


In [64]:
# Fillet slanted bottom edge of shaft and throughhole
targetFaces = []
filletItems = []

for i, x in enumerate(shaftListWithThroughHoles):
    targetEdges = []
    targetFace = min(shaftListWithThroughHoles[i].faces(), key = findZ)
    targetFaces.append(targetFace)
    targetEdge1, targetEdge2 = targetFace.edges()
    targetEdges.append(targetEdge1)
    targetEdges.append(targetEdge2)
    filletItem: Sketch | Part | Curve = fillet(targetEdges, radius = 0.20)
    filletItems.append(filletItem)

shaftListWithThroughHoles = filletItems

show(targetEdges, shaftListWithThroughHoles, reset_camera=Camera.RESET)

cccc


In [65]:
# Fillet-ing top edge of throughholes
filletList = []

for i, x in enumerate(shaftListWithThroughHoles):
    edge = x.edges().filter_by(GeomType.CIRCLE).filter_by(
        lambda a: abs(a.radius - innerShaftRadius) < 1e-6)
    seams.append(edge)

    filletObject = fillet(edge, radius = 0.25)
    filletList.append(filletObject)

show(filletList, reset_camera = Camera.RESET)
shaftListWithThroughHoles = filletList


cccc


In [66]:
# Display everything for the purpose of sanity checks

show(basisCylinder, shaftListWithThroughHoles, chamberMold, alphas=[1.0,1.0,0.8], colors=["#e8b024", "#e8b024", "lightblue"], reset_camera=Camera.RESET)

cccccc


Okay, the actual through holes for the GT have been implanted in the shafts. Location-dependant shaft placement and homing has been completed in a scalable fashion. Now we need to connect the shafts to the basis cylinder. 

In [67]:
# GT Depth Calculator: To-do

In [68]:
# Nubs
# Doing this a little bit differently from Anna's Onshape. Instead of building in the default plane, I'm going to build each set of nubs on the top plane of the actual shaft it's attached to.

parallelNubSeparation = 2.64575
nubLongSide = 3
nubShortSide = 1.35425
nubDepth = 2.5
nubsList = []
rotatorLocations = [1,2,3,4]
rotatorAngle = 90
overlaps = []
nubTemplates = []
nubTemplatesFlattened = []

nubDisplacementVector = (0,parallelNubSeparation/2 + nubShortSide/2,0)
nubRotationVector = (0,0,1)

for i, x in enumerate(startingOffsetPlanes):
    for k, j in enumerate(rotatorLocations):
        nubTemplate = Rectangle(nubLongSide,nubShortSide).located(Location(startingOffsetPlanes[i])).translate(nubDisplacementVector).rotate(
            axis = Axis(startingOffsetPlanes[i].origin, nubRotationVector),
            angle = rotatorAngle*k
        )
        nubTemplates.append(nubTemplate)
        with BuildPart() as nubs:
            extrude(nubTemplate, amount=nubDepth, dir = (0,0,-1))
        nubsList.append(nubs.part)

for i, x in enumerate(nubTemplates):
    nubTemplatesFlattened.append(flattenToXY(nubTemplates[i]))

# Overlap areas between nubs 2,4
overlaps.append(nubTemplatesFlattened[2] & nubTemplatesFlattened[4])

# Overlap areas between nubs 6,8
overlaps.append(nubTemplatesFlattened[6] & nubTemplatesFlattened[8])

# Overlap areas between nubs 8,10
overlaps.append(nubTemplatesFlattened[10] & nubTemplatesFlattened[12])

print(overlaps)

show(basisCylinder, shaftListWithThroughHoles, nubsList, overlaps, reset_camera=Camera.RESET)


[Compound at 0x1210f49f0b0, label(), #children(0), Compound at 0x1210f49d7f0, label(), #children(0), Compound at 0x1210f49c3b0, label(), #children(0)]
cccccccccccccccccccccccc


In [69]:
# Constructing overlap solids, pruning union parts between different shafts

overlapSolids = []
prunedParts = []
lofts = []

# Generating overlap structures
for i, x in enumerate(overlaps):
    with BuildPart() as firstpart:
        extrude(overlaps[i], amount = shaftHeight*10, dir = (0,0,-1))
        overlapSolids.append(firstpart.part)

# Deleting overlap between shafts 1 and 2, numbered from +y to -y in global coordinates
with BuildPart() as pt:
    add (nubsList[2])
    add (overlapSolids[0], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

with BuildPart() as pt:
    add (nubsList[4])
    add (overlapSolids[0], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

loft1 = loftMe(nubsList[2], nubsList[4])
lofts.append(loft1)

# Deleting overlap between shafts 10 and 12, numbered from +y to -y in global coordinates
with BuildPart() as pt:
    add (nubsList[10])
    add (overlapSolids[2], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

with BuildPart() as pt:
    add (nubsList[12])
    add (overlapSolids[2], mode = Mode.SUBTRACT)
    prunedParts.append(pt.part)

loft2 = loftMe(nubsList[10], nubsList[12])
lofts.append(loft2)

with BuildPart() as pt:
    if startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
        add (nubsList[6])
    elif startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
        add (nubsList[8])
    else:
        add (nubsList[6])
        add (nubsList[8])
    add(overlapSolids[1], mode=Mode.SUBTRACT)
    prunedParts.append(pt.part)

show(prunedParts, reset_camera=Camera.RESET, colors=["#e8b024", "#e8b024", "#e8b024",  "lightblue"])


Too many colors, trimming to length 1
ccccc


In [70]:
show(prunedParts[-1])
len(prunedParts)

c


5

In [71]:
# Combining nubs and shafts with throughholes

nubulousShaftsWithThroughHoles = []
iteratorNumbers = []

for i, x in enumerate(shaftListWithThroughHoles):
    with BuildPart() as NubulousShaftWithThroughHole:
        for k, j in enumerate(nubsList):
            iteratorNumber = k // 4
            if iteratorNumber == i and j != 6 and j != 8:
                add(nubsList[k])
            elif iteratorNumber == i and j == 6 or iteratorNumber == i and j == 8:
                if startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
                    add (nubsList[6])
                elif startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
                    add (nubsList[8])
                else:
                    add (nubsList[6])
                    add (nubsList[8])
            else:
                continue
        add(shaftListWithThroughHoles[i])
        add(overlapSolids, mode = Mode.SUBTRACT)
    nubulousShaftsWithThroughHoles.append(NubulousShaftWithThroughHole.part)

show(nubulousShaftsWithThroughHoles, reset_camera=Camera.RESET)




# Pseudocode
# add the 4 adjacent nubs to their shaft
    # add the shaft i
    # add the 4 nubs whose floor division is equal to i
    # Combine these into single part
# fillet the intersections
# append these combined units to a new list

++++


In [72]:
# type Part = build123d.topology.composite.Part
from build123d.topology.composite import Part

In [73]:
def printPart(part: Part):
    print(part)

printPart(nubsList[0])
printPart(3)

Part at 0x1210f49dd90, label(), #children(0)
3


In [74]:
print(type(nubsList[1]))

<class 'build123d.topology.composite.Part'>


In [75]:
# Generating positional arms to connect nubs and basis cylinder

testArms = []
outputArmStream = []
refinedNubsList = [nubsList[1], nubsList[5], nubsList[9], nubsList[13]]

for i, x in enumerate(refinedNubsList):
    arm1, mirrorArm,armTest = armConstructor(i, refinedNubsList[i],startingOffsetPlanes[i])
    outputArmStream.append(arm1)
    outputArmStream.append(mirrorArm)
    testArms.append(armTest)
    # Ensure construction sketches are either nub-centric, or somehow related directionally to the rotation of the arbor centerline. 
show(basisCylinder, shaftListWithThroughHoles, nubsList, outputArmStream, colors=["#e8b024", "#e8b024", "#e8b024",  "lightblue", "pink", "pink"])



Too many colors, trimming to length 4
c++++++++++c+c+++++++cccccccc


In [76]:
# Pre-fusion chamfers. Use geometry to isolate faces and identify edges as join points between two faces. Merge internal edges this way
input = basisCylinder
# all this will be need to reworked for use with rotational planes and the like. for now it works fine 
xNormalFaces, zNormalFaces = obtainFaces(basisCylinder)
faceList = basisCylinder.faces()

targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]

vIndicies = [9,3,6,4]
zIndicies = [0,1,1,0]
rads = [2.9,0.9,0.9,2.9]
# Compacted code 
input = basisCylinder
"""
for i, x in enumerate(zIndicies):
    xNormalFaces, zNormalFaces = obtainFaces(input)
    faceList=input.faces()
    targetZFace = [zNormalFaces[0], zNormalFaces[1]]
    show(faceList[vIndicies[i]],targetZFace[zIndicies[i]],input)
    time.sleep(0.5)
    sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
    input = fillet(sharedEdges, radius = rads[i])
show(input)
"""

show(faceList[3], targetZFace[zIndicies[2]])
# Expanded code. Going to have to just figure out the fillets 1 by 1 with the updated geometries in order to find proper indicies
# for intermediate geometry stages
i = 0
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = fillet(sharedEdges, radius = rads[i])
show(input)
# Hypothesis: z faces are not changing, but vfaces is 
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
show(faceList[3], targetZFace[zIndicies[2]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = fillet(sharedEdges, radius = rads[i])
show(input)
# slice 3
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
show(faceList[6], targetZFace[zIndicies[2]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = fillet(sharedEdges, radius = rads[i])
show(input)
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = fillet(sharedEdges, radius = rads[i])
show(input)

basisCylinderModified = input

++
+
cc
+
cc


c
+


In [77]:
i = 0
# Dupicate for testing purposes
# Pre-fusion chamfers. Use geometry to isolate faces and identify edges as join points between two faces. Merge internal edges this way
input = basisCylinder
# all this will be need to reworked for use with rotational planes and the like. for now it works fine 
xNormalFaces, zNormalFaces = obtainFaces(basisCylinder)
faceList = basisCylinder.faces()

targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]


vIndicies = [9,6,6,8]
zIndicies = [0,0,0,0]
rads = [1,1,6,6]
# Compacted code 
input = basisCylinder
for i, x in enumerate(zIndicies):
    if i == 2 or i == 3:
        xNormalFaces, zNormalFaces = obtainFaces(input)
        faceList=input.faces()
        targetZFace = [zNormalFaces[0], zNormalFaces[1]]
        show(faceList[vIndicies[i]],targetZFace[zIndicies[i]],input)
        time.sleep(0.25)
        sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
        input = fillet(sharedEdges, radius = rads[i])
    else:
        xNormalFaces, zNormalFaces = obtainFaces(input)
        faceList = input.faces()
        targetZFace = [zNormalFaces[0], zNormalFaces[1]]
        show(faceList[vIndicies[i]], targetZFace[zIndicies[i]])
        time.sleep(0.25)
        sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
        input = chamfer(sharedEdges, length = 8, length2 = 3.99999, reference = targetZFace[zIndicies[i]])
        show(input)
"""

# Expanded code. Going to have to just figure out the fillets 1 by 1 with the updated geometries in order to find proper indicies
# for intermediate geometry stages
i = 0
show(faceList[vIndicies[i]], targetZFace[zIndicies[i]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = chamfer(sharedEdges, length = 8, length2 = 3.99999, reference = targetZFace[zIndicies[i]])
show(input)
# Hypothesis: z faces are not changing, but vfaces is 
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
show(faceList[vIndicies[i]], targetZFace[zIndicies[i]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = chamfer(sharedEdges, length = 8, length2 = 3.99999, reference = targetZFace[zIndicies[i]])
show(input)
# slice 3
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
show(faceList[vIndicies[i]], targetZFace[zIndicies[i]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
input = fillet(sharedEdges, radius = rads[i])
show(input)
# slice3 4
xNormalFaces, zNormalFaces = obtainFaces(input)
faceList = input.faces()
targetVFace = [faceList[9], faceList[11]]
targetZFace = [zNormalFaces[0], zNormalFaces[1]]
i = i + 1
show(faceList[vIndicies[i]], targetZFace[zIndicies[i]])
sharedEdges = faceList[vIndicies[i]].edges() & targetZFace[zIndicies[i]].edges()
print(sharedEdges)
input = fillet(sharedEdges, radius = rads[i])
show(input)
"""
"""
"""
basisCylinderModified = input
show(basisCylinderModified)

++
+
cc
c
+cc
+c+
+


In [78]:
# probably make this a function
for i, x in enumerate(faceList):
    show(faceList[i], targetZFace[zIndicies[3]])

cc
cc
cc
cc
-c
cc
cc
cc
cc
cc
cc
cc
cc
cc
cc
cc
cc
cc


In [79]:
# Site-based data storage 

@dataclass
class Site:
    shaft: Part
    plane: Plane
    nubs: list
    arms: list = field(default_factory = list)
    outer: object = None
    throughHole: object = None
    combined: object = None

# Establishing the variables used in this cell
sites = []
planeHeightIndicator = 0

# handles the x-axial nubs and the outer union parts, plus the shafts
for i, plane in enumerate(startingOffsetPlanes):
    sites.append(Site(
        shaft = shaftListWithThroughHoles[i],
        plane = plane,
        nubs = nubsList[i*4+1:i*4 + 4:2] + [prunedParts[i]],
        arms = outputArmStream[i*2:i*2 + 2]
    )) 

# Picks the highest(z) starting offset plane of the two middle planes. 
# Adds the central nub of the higher z site and appends to nubs
# trims the central nub of the lower z site and appends that to nubs
if startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
    sites[1].nubs.append((nubsList[6]))
    sites[2].nubs.append((prunedParts[-1]))
    planeHeightIndicator = 2 # this keeps track of the higher plane
elif startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
    sites[2].nubs.append((nubsList[8]))
    sites[1].nubs.append((prunedParts[-1]))
    planeHeightIndicator = 1 # this keeps track of the higher plane

# covers the case in which the zaxis is the same, in which case we just add both nubs
else:
    sites[1].nubs.append((nubsList[6]))
    sites[2].nubs.append((nubsList[8]))

show(*[site.shaft for site in sites], [site.nubs for site in sites], [site.arms for site in sites])

cccccc+cc++cc+ccc+cccccccc


In [80]:
# Constructing final solid part
with BuildPart() as fusedPart:

    # Shafts and side nubs
    for i, x in enumerate(shaftListWithThroughHoles):
        add (shaftListWithThroughHoles[i]), 
        add (nubsList[1::2])
        """
        """
    # separable npx pairs need a union. this is the math for that union.
    add (prunedParts[4])
    if startingOffsetPlanes[1].origin.Z < startingOffsetPlanes[2].origin.Z:
        add (nubsList[6])
    elif startingOffsetPlanes[1].origin.Z > startingOffsetPlanes[2].origin.Z:
        add (nubsList[8])
    else:
        add (nubsList[6], nubsList[8])
    
    # paired npx nubs
    add (prunedParts[0])
    add (prunedParts[1])
    add (loft1)

    add (prunedParts[2])
    add (prunedParts[3])
    add (loft2)

with BuildPart() as anotherpart:
    # Basis cylinder
    add (basisCylinderModified)
    add (fusedPart)

    # adding arms
    add (outputArmStream)
    


guideTubeFrameNoChamfer = anotherpart
show(guideTubeFrameNoChamfer, basisCylinderModified, reset_camera=Camera.RESET)

++


In [81]:
# Sitewise construction using dataclasses

fillets = []
filletedItems = []


for i, site in enumerate(sites):
    with BuildPart() as combinedSite:
        add(site.shaft)
        add(site.nubs)
    filletMe = new_edges(
        site.shaft, 
        *site.nubs, 
        combined = combinedSite.part).filter_by(GeomType.LINE)
    fillets.extend(filletMe)
    if combinedSite.part.max_fillet(filletMe) > 0.25:
        filletParameter = 0.25
    else:
        filletParameter = combinedSite.part.max_fillet(filletMe)

    filletedItem = fillet(filletMe, radius = filletParameter)
    
    filletedItems.append(filletedItem)



show(filletedItems)

cccc


In [82]:
# Adding arms sitewise
combinedSitesWithArms = []


for i, site in enumerate(sites):
    with BuildPart() as combinedSiteWithArms:
        add(filletedItems[i])
        add(site.arms)
        combinedSitesWithArms.append(combinedSiteWithArms.part)

show([combinedSitesWithArms for site in sites])


------------++++


In [83]:
# Merging with BasisCylinder

objectsInProgress = [basisCylinderModified]
armFillets = []

for i, site in enumerate(sites):
    with BuildPart() as newPart:
        add(objectsInProgress[i])
        add(combinedSitesWithArms[i])
        armFillet = new_edges(
            combinedSitesWithArms[i],
            objectsInProgress[i],
            combined = newPart.part).filter_by(GeomType.LINE).filter_by(lambda e: e.length >= 2.25)
        objectsInProgress.append(fillet(armFillet, radius = 0.15))
        armFillets.append(armFillet)

show(*[armFillets], objectsInProgress[-1])
armFillet[2].length


+


2.500000000000001

In [84]:
# adding lofts to make final part  
objectsInProgress2 = list(objectsInProgress)
loftFillets = []

with BuildPart() as nextPart:
    add (objectsInProgress2[-1])
    add (loft1)
    add (loft2)
    """loftFillet = new_edges(
        loft,
        objectsInProgress2[-1],
        combined = newPart.part).filter_by(GeomType.LINE)"""
    show(objectsInProgress2,loft)
    loftFillet = nextPart.edges().filter_by(
        GeomType.LINE).filter_by(
        lambda e: abs(e.length - 3) < 1e-6).filter_by(
        lambda e: abs(e.position_at(0).Z - e.position_at(1).Z) < 1e-6).filter_by(
        lambda e: abs(e.position_at(0).Y - e.position_at(1).Y) < 1e-6).filter_by(
        lambda e: e.center().Z < -1)
    loftFillets.extend(loftFillet)
    objectsInProgress.append(nextPart.part)
    print(loftFillets)
    objectsInProgress2.append(fillet(loftFillet, radius = 0.1))
        
finalPart = objectsInProgress2[-1]
show(loftFillets, finalPart)

cc+cc
[<build123d.topology.one_d.Edge object at 0x000001209012E5F0>, <build123d.topology.one_d.Edge object at 0x000001210F4F4A60>, <build123d.topology.one_d.Edge object at 0x000001210F4F7380>, <build123d.topology.one_d.Edge object at 0x000001210F4F6430>, <build123d.topology.one_d.Edge object at 0x000001209723DFD0>, <build123d.topology.one_d.Edge object at 0x000001209723C750>, <build123d.topology.one_d.Edge object at 0x000001208CCCE510>, <build123d.topology.one_d.Edge object at 0x000001210F522F20>]
+


In [143]:
# Generating threaded screw inserts / holes

threadDepth = 3.5
threadDiameter = 3.0
threadPitch = 0.5
interferenceConstant = 0.15

ridge = bd_warehouse.thread.IsoThread(
    major_diameter = threadDiameter,
    pitch = threadPitch,
    length = threadDepth,
    external = False,
    interference = interferenceConstant,
    align = (Align.CENTER, Align.CENTER, Align.MIN)
)

outerCylinder = Cylinder(radius = threadDiameter/2, height = threadDepth, align = (Align.CENTER, Align.CENTER, Align.MIN))

threadedInsert = outerCylinder - ridge

threadedInsertMoveVector = (0,0,-1)

threadedInsertMoved = threadedInsert.translate(threadedInsertMoveVector)

threadedInsertRotationVector = (1,0,0)
threadedInsertTranslationVector = (0, -3, 0)

show(threadedInsert, finalPart, reset_camera=Camera.RESET)
time.sleep(0.5)

threadedInsertRotated = threadedInsertMoved.rotate(
    axis = Axis(Vector(0,0,0), threadedInsertRotationVector),
    angle = -45
).translate(threadedInsertTranslationVector)

show(threadedInsertRotated, finalPart, reset_camera=Camera.RESET)

+c
+c


In [144]:
# Generating 45 degree screwholes

# constant d = 3 + 17.74824
displacementNumber = 22.74824 - 18 # This float comes from Onshape. Center of chamber to edge of cube. It's subtracted from the initial chamberradius3 value in a variable agnostic fashion.
# essentially the above is a constant offset which can be used with any chamber radius size to appropriately locate the edge cubes.
print(displacementNumber)
cubeYDisplacement = displacementNumber + chamberRadius3
cubeZDisplacement = -3
cubeXDisplacement = 0
centralRotatorAngle = -15
iteratorRotatorAngle = 120


newCube = generateCube(6,6,6)
show(finalPart, newCube)

cubeEdgesForChamfer = newCube.edges().filter_by(GeomType.LINE).filter_by(lambda e: e.center().Z > -.1).filter_by(lambda e: e.center().Y > -.1).filter_by(lambda e: abs(e.position_at(0).Z - e.position_at(1).Z) < 1e-6)
show(cubeEdgesForChamfer, newCube, finalPart)

newChamferedCube = chamfer(cubeEdgesForChamfer, length = 3)
show(newChamferedCube,threadedInsertRotated, finalPart)
time.sleep(10)


# Translation and rotation vectors
cubeTranslationVector = (cubeXDisplacement, cubeYDisplacement, cubeZDisplacement)
cubeRotationVector = (0,0,1)

newChamferedCubeWithScrewHole = newChamferedCube - threadedInsertRotated

show(newChamferedCubeWithScrewHole, finalPart)

translatedCube = newChamferedCubeWithScrewHole.translate(cubeTranslationVector)
show(translatedCube, finalPart)

translatedRotatedCube = translatedCube.rotate(
    axis = Axis(Vector(0,0,0), cubeRotationVector),
    angle = centralRotatorAngle)

translatedRotatedCube2 = translatedCube.rotate(
    axis = Axis(Vector(0,0,0), cubeRotationVector),
    angle = centralRotatorAngle + iteratorRotatorAngle)

translatedRotatedCube3 = translatedCube.rotate(
    axis = Axis(Vector(0,0,0), cubeRotationVector),
    angle = centralRotatorAngle + iteratorRotatorAngle*2)

show(translatedRotatedCube, translatedRotatedCube2, translatedRotatedCube3, finalPart)

4.748239999999999
cc
cc
c+c
+c
+c
+++c


In [146]:
# Merge

with BuildPart() as finalPartWithScrewHoles:
    add(finalPart)
    add(translatedRotatedCube)
    add(translatedRotatedCube2)
    add(translatedRotatedCube3)

show(finalPartWithScrewHoles, reset_camera=Camera.RESET)

+


In [ ]:
show(loftFillets, objectsInProgress2[-1])

c


In [ ]:
for i, e in enumerate(loftFillet):
    try:
        r = nextPart.part.max_fillet([e])
        print(i, "ok — max radius:", r)
    except Exception as err:
        print(i, "BAD EDGE:", type(err).__name__, err)


0 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
1 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
2 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
3 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
4 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
5 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
6 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet
7 BAD EDGE: Standard_Failure There are no suitable edges for chamfer or fillet


In [ ]:

print(len(nextPart.part.solids()))


1


In [ ]:
export_stl(finalPart, "ParametricGuideTubeFrame.stl")

True